# Intro to PandaPower for IDP Project Optimization

In [1]:
try:
    from pandapower.networks.power_system_test_cases import case30
    from pandapower.run import runpp, runopp
    from pandapower.timeseries.run_time_series import run_timeseries
except: 
    %pip install pandapower
    %pip install numba
    from pandapower.networks.power_system_test_cases import case30
    from pandapower.run import runpp, runopp
    from pandapower.timeseries.run_time_series import run_timeseries

from tools.limits import bus_vm_pu_limits, line_loading_limits
import profileloader as pl
import numpy as np

### Import IEEE Case 30:
Case 30 is stored in a variable, "net", which consists of several dataframes. 
You can conceptualize a dataframe as being like a spreadsheet: It has named rows and columns, and data is stored in individual cells for each row/column pair. 
By default, the IEEE 30 net has a dataframe for each of the following grid components
- bus (30)
- load (21)
- generator (5)
- shunt (2)
- ext_grid (1)
- line (34)
- transformer (7)
- cost (6)

Additional components, such as for storage, motors, etc. can also be added to the net

In [2]:
net = case30()
print(net)

This pandapower network includes the following parameter tables:
   - bus (30 elements)
   - load (20 elements)
   - gen (5 elements)
   - shunt (2 elements)
   - ext_grid (1 element)
   - line (41 elements)
   - poly_cost (6 elements)


In [3]:
net.poly_cost

,element,et,cp0_eur,cp1_eur_per_mw,cp2_eur_per_mw2,cq0_eur,cq1_eur_per_mvar,cq2_eur_per_mvar2
0,0,ext_grid,0.0,2.00,0.02000,0.0,0.0,0.0
1,0,gen,0.0,1.75,0.01750,0.0,0.0,0.0
2,1,gen,0.0,1.00,0.06250,0.0,0.0,0.0
3,2,gen,0.0,3.25,0.00834,0.0,0.0,0.0
4,3,gen,0.0,3.00,0.02500,0.0,0.0,0.0
5,4,gen,0.0,3.00,0.02500,0.0,0.0,0.0


In [4]:
net.load.iloc[0:5,]

,name,bus,p_mw,q_mvar,const_z_p_percent,const_z_q_percent,const_i_p_percent,const_i_q_percent,sn_mva,scaling,in_service,type,controllable
0,None,1,21.7,12.7,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False
1,None,2,2.4,1.2,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False
2,None,3,7.6,1.6,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False
3,None,6,22.8,10.9,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False
4,None,7,30.0,30.0,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False


Specific cells are adjustable based on external data

In [5]:
net.line

,name,std_type,from_bus,to_bus,length_km,r_ohm_per_km,x_ohm_per_km,c_nf_per_km,g_us_per_km,max_i_ka,df,parallel,type,in_service,max_loading_percent,geo
0,None,None,0,1,1.0,3.6450,10.9350,436.639076,0.0,0.555967,1.0,1,ol,True,100.0,None
1,None,None,0,2,1.0,9.1125,34.6275,291.092717,0.0,0.555967,1.0,1,ol,True,100.0,None
2,None,None,1,3,1.0,10.9350,30.9825,291.092717,0.0,0.277983,1.0,1,ol,True,100.0,None
3,None,None,2,3,1.0,1.8225,7.2900,0.000000,0.0,0.555967,1.0,1,ol,True,100.0,None
4,None,None,1,4,1.0,9.1125,36.4500,291.092717,0.0,0.555967,1.0,1,ol,True,100.0,None
5,None,None,1,5,1.0,10.9350,32.8050,291.092717,0.0,0.277983,1.0,1,ol,True,100.0,None
6,None,None,3,5,1.0,1.8225,7.2900,0.000000,0.0,0.384900,1.0,1,ol,True,100.0,None
7,None,None,4,6,1.0,9.1125,21.8700,145.546359,0.0,0.299367,1.0,1,ol,True,100.0,None
8,None,None,5,6,1.0,5.4675,14.5800,145.546359,0.0,0.555967,1.0,1,ol,True,100.0,None
9,None,None,5,7,1.0,1.8225,7.2900,0.000000,0.0,0.136853,1.0,1,ol,True,100.0,None


In [6]:
# for example: set the power consumption of load #1 to 2x its starting value
print("Before: " + str(net.load.at[0,'p_mw']))
net.load.at[0,'p_mw'] = 2 * net.load.iloc[0].p_mw
print("After: " + str(net.load.at[0,'p_mw']))

# and: set line loading limit to 50%
print("Before: " + str(net.line.at[10,'max_loading_percent']))
net.line.at[10,'max_loading_percent'] = 0.5
print("After: " + str(net.line.at[10,'max_loading_percent']))

Before: 21.7
After: 43.4
Before: 100.0
After: 0.5


### Power flow analysis
Analysis can be run on net using the runpp function. This adds a new set of dataframes to the net, containing result data for each bus, load, generator, etc.

In [7]:
runpp(net)
net.res_bus[0:5]

,vm_pu,va_degree,p_mw,q_mvar
0,1.000000,0.000000,-47.846249,7.653593
1,1.000000,-1.112025,-17.570000,-26.597780
2,0.983135,-1.940142,2.400000,1.200000
3,0.980162,-2.302209,7.600000,1.600000
4,0.982396,-2.490312,0.000000,-0.183369


In [8]:
net.res_gen

,p_mw,q_mvar,va_degree,vm_pu
0,60.97,39.297780,-1.112025,1.0
1,21.59,39.554809,-3.932185,1.0
2,26.91,10.527570,-1.374166,1.0
3,19.20,7.938512,-2.121219,1.0
4,37.00,11.319832,0.951621,1.0


You can use the result dataframes to check if power flow data is within the defined limits for each object

In [9]:
# For example, buses:
bus_vm_pu_limits(net)
line_loading_limits(net)

All buses within limits
Line #9: Result 111.8198% loading is greater than maximum of 100.0% loading
Line #10: Result 10.4801% loading is greater than maximum of 0.5% loading


For purposes of optimization, grid components such as generators and loads have a boolean property called "controllable". If a component's "controllable" property is true, that means that the component can be changed during the optimization algorithm, to find which values produce the optimal grid conditions. Values where "controllable" is false are those which must be kept fixed.  
By default, all sources are controllable, and all loads are not. 

In [10]:
runopp(net)

Below, you can see that the controllable grids have had their p_mw values changed. Compare the generator results with the generator results from before:

In [11]:
net.res_gen

,p_mw,q_mvar,va_degree,vm_pu
0,53.623339,25.937628,-0.964218,1.001399
1,28.286298,7.339579,-1.613982,0.994606
2,44.732117,29.905295,1.609648,1.068951
3,24.099484,9.353275,-0.788258,1.034163
4,22.954928,44.698565,-0.772041,1.086644


In [12]:
# proof that bus max/min values are now accounted for
bus_vm_pu_limits(net)
line_loading_limits(net)

All buses within limits
All line loading within limits


To demonstrate optimization working if we set one of the generators to a fixed, non-controllable value: 

In [13]:
net.gen.at[0,'p_mw'] = 20
net.gen.at[0, 'controllable'] = False
net.gen

,name,bus,p_mw,vm_pu,sn_mva,min_q_mvar,max_q_mvar,scaling,slack,in_service,slack_weight,type,controllable,max_p_mw,min_p_mw,id_q_capability_characteristic,reactive_capability_curve,curve_style
0,None,1,20.00,1.0,NaN,-20.0,60.0,1.0,False,True,0.0,None,False,80.0,0.0,<NA>,False,None
1,None,21,21.59,1.0,NaN,-15.0,62.5,1.0,False,True,0.0,None,True,50.0,0.0,<NA>,False,None
2,None,26,26.91,1.0,NaN,-15.0,48.7,1.0,False,True,0.0,None,True,55.0,0.0,<NA>,False,None
3,None,22,19.20,1.0,NaN,-10.0,40.0,1.0,False,True,0.0,None,True,30.0,0.0,<NA>,False,None
4,None,12,37.00,1.0,NaN,-15.0,44.7,1.0,False,True,0.0,None,True,40.0,0.0,<NA>,False,None


In [14]:
runpp(net)
bus_vm_pu_limits(net)
line_loading_limits(net)

All buses within limits
Line #9: Result 111.7991% loading is greater than maximum of 100.0% loading
Line #10: Result 10.3021% loading is greater than maximum of 0.5% loading


In [15]:
runopp(net)
bus_vm_pu_limits(net)
line_loading_limits(net)


All buses within limits
All line loading within limits


The resulting optimized components have changed based on this new constraint. 

In [16]:
net.gen


,name,bus,p_mw,vm_pu,sn_mva,min_q_mvar,max_q_mvar,scaling,slack,in_service,slack_weight,type,controllable,max_p_mw,min_p_mw,id_q_capability_characteristic,reactive_capability_curve,curve_style
0,None,1,20.00,1.0,NaN,-20.0,60.0,1.0,False,True,0.0,None,False,80.0,0.0,<NA>,False,None
1,None,21,21.59,1.0,NaN,-15.0,62.5,1.0,False,True,0.0,None,True,50.0,0.0,<NA>,False,None
2,None,26,26.91,1.0,NaN,-15.0,48.7,1.0,False,True,0.0,None,True,55.0,0.0,<NA>,False,None
3,None,22,19.20,1.0,NaN,-10.0,40.0,1.0,False,True,0.0,None,True,30.0,0.0,<NA>,False,None
4,None,12,37.00,1.0,NaN,-15.0,44.7,1.0,False,True,0.0,None,True,40.0,0.0,<NA>,False,None


In [17]:
net.res_line

,p_from_mw,q_from_mvar,p_to_mw,q_to_mvar,pl_mw,ql_mvar,i_from_ka,i_to_ka,i_ka,vm_from_pu,va_from_degree,vm_to_pu,va_to_degree,loading_percent
0,43.957613,-15.443639,-43.532274,13.719657,4.253394e-01,-1.723982,0.199257,0.195200,0.199257,1.000000,0.000000,1.000000,-1.671171,35.839697
1,15.439068,0.626980,-15.318562,-2.148366,1.205059e-01,-1.521386,0.066082,0.066849,0.066849,1.000000,0.000000,0.989600,-1.651520,12.023875
2,4.735411,4.438552,-4.704210,-6.326144,3.120118e-02,-1.887592,0.027757,0.034127,0.034127,1.000000,-1.671171,0.987925,-1.948803,12.276717
3,12.918564,0.948366,-12.901430,-0.879833,1.713338e-02,0.068534,0.055979,0.055979,0.055979,0.989600,-1.651520,0.987925,-1.948803,10.068811
4,7.992213,3.980393,-7.947873,-5.775497,4.433989e-02,-1.795104,0.038184,0.042608,0.042608,1.000000,-1.671171,0.986135,-2.455228,7.663728
5,7.404652,6.598837,-7.337109,-8.360373,6.754272e-02,-1.761536,0.042417,0.048447,0.048447,1.000000,-1.671171,0.981918,-2.182860,17.427977
6,12.819385,11.649953,-12.788641,-11.526978,3.074379e-02,0.122975,0.074987,0.074987,0.074987,0.987925,-1.948803,0.981918,-2.182860,19.482121
7,7.947874,5.960265,-7.894028,-6.791879,5.384575e-02,-0.831615,0.043084,0.045712,0.045712,0.986135,-2.455228,0.974282,-2.831778,15.269422
8,14.980366,3.349811,-14.905972,-4.108121,7.439444e-02,-0.758310,0.066857,0.067870,0.067870,0.981918,-2.182860,0.974282,-2.831778,12.207580
9,21.134492,23.251489,-21.032093,-22.841891,1.023995e-01,0.409598,0.136853,0.136853,0.136853,0.981918,-2.182860,0.970314,-2.551415,99.999733


In [18]:
#const_load = pl.csv_to_net(net, "load")

In [ ]:
load_control = pl.json_to_net(net, "load")
gen_conrol = pl.json_to_net(net, "gen")

In [21]:
print(net.gen)

   name  bus       p_mw  vm_pu  sn_mva  min_q_mvar  max_q_mvar  scaling  \
0  None    1  -1.069760    1.0     NaN       -20.0        60.0      1.0   
1  None   21  -5.348801    1.0     NaN       -15.0        62.5      1.0   
2  None   26   0.000000    1.0     NaN       -15.0        48.7      1.0   
3  None   22   0.000000    1.0     NaN       -10.0        40.0      1.0   
4  None   12 -10.697601    1.0     NaN       -15.0        44.7      1.0   

   slack  in_service  slack_weight  type  controllable  max_p_mw  min_p_mw  \
0  False        True           0.0  None         False      80.0       0.0   
1  False        True           0.0  None          True      50.0       0.0   
2  False        True           0.0  None          True      55.0       0.0   
3  False        True           0.0  None          True      30.0       0.0   
4  False        True           0.0  None          True      40.0       0.0   

   id_q_capability_characteristic  reactive_capability_curve curve_style  
0    

In [20]:
run_timeseries(net)

No time steps to calculate are specified. I'll check the datasource of the first controller for avaiable time steps
  0%|          | 0/35401 [00:00<?, ?it/s]CalculationNotConverged at time step 0


LoadflowNotConverged: 